In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from master_mind.teaching.hf import HFModel


In [ ]:
def get_best_device():
    """Returns the best device on this computer"""

    if torch.cuda.is_available():
        device = torch.device("cuda")
        total_memory = torch.cuda.get_device_properties(device).total_memory
        print(f"GPU Memory: {total_memory / 1e9:.1f} GB")
        print(f"GPU Name: {torch.cuda.get_device_name(device)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Found device: {device}")
    return device


device = get_best_device()

## Environmental Impact

Training and running LLMs has environmental costs. In this exercise,
we'll measure CO2 emissions and understand trade-offs between model size
and environmental impact.

### Exercise 4.1: Track Emissions Across Model Sizes

We'll use `codecarbon` to track CO2 emissions for different model sizes.

### Environmental Impact Measurement

In [ ]:
# Import codecarbon
import platform

# On macOS, patch codecarbon to skip powermetrics (requires sudo)
if platform.system() == "Darwin":
    import codecarbon.core.powermetrics as pm

    pm.is_powermetrics_available = lambda: False

from codecarbon import OfflineEmissionsTracker

print("Using offline emissions tracker (estimated carbon intensity)")


def get_tracker(project_name: str) -> OfflineEmissionsTracker:
    """Create an emissions tracker."""
    # Offline mode with France carbon intensity (~50 gCO2/kWh)
    return OfflineEmissionsTracker(
        project_name=project_name,
        country_iso_code="FRA",
        log_level="error",
    )

In [ ]:
# Models to compare (from smallest to largest)
MODELS_TO_COMPARE = {
    "DistilGPT-2 (82M)": HFModel(
        "distilgpt2", tokenizer_cls=GPT2Tokenizer, model_cls=GPT2LMHeadModel
    ),
    "GPT-2 Small (124M)": HFModel(
        "gpt2", tokenizer_cls=GPT2Tokenizer, model_cls=GPT2LMHeadModel
    ),
    "GPT-2 Medium (355M)": HFModel(
        "gpt2-medium", tokenizer_cls=GPT2Tokenizer, model_cls=GPT2LMHeadModel
    ),
}



def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 50) -> str:
    """Generate text from a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Test prompts
test_prompts = [
    "The future of artificial intelligence is",
    "Climate change is affecting",
    "The most important thing in life is",
]

# Number of generations per model
n_generations = 10


# Track emissions for each model
results = []

for model_label, hf_model in MODELS_TO_COMPARE.items():
    print(f"\nTesting {model_label}...")

    # Load model and tokenizer (cached properties)
    tokenizer = hf_model.tokenizer
    model = hf_model.model
    model = model.to(device)
    model.eval()

    # Set pad token
    tokenizer.pad_token = tokenizer.eos_token

    # Track emissions
    tracker = get_tracker(project_name=model_label)
    tracker.start()

    for _ in range(n_generations):
        prompt = np.random.choice(test_prompts)
        _ = generate_text(model, tokenizer, prompt)

    emissions = tracker.stop()

    # Get model size
    n_params = sum(p.numel() for p in model.parameters())

    results.append(
        {
            "model": model_label,
            "params_millions": n_params / 1e6,
            "emissions_g": emissions * 1000,  # Convert kg to g
            "emissions_per_gen_mg": (emissions * 1e6) / n_generations,
        }
    )

    # Free memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Display results
df_emissions = pd.DataFrame(results)
print("\n" + "=" * 60)
print("Emissions Comparison")
print("=" * 60)
print(df_emissions.to_string(index=False))

### Exercise 4.2: Visualize the Trade-offs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Emissions vs Model Size
ax1 = axes[0]
ax1.bar(
    df_emissions["model"], df_emissions["emissions_per_gen_mg"], color="forestgreen"
)
ax1.set_xlabel("Model")
ax1.set_ylabel("CO2 Emissions per Generation (mg)")
ax1.set_title("Carbon Footprint by Model Size")
ax1.tick_params(axis="x", rotation=15)

# Plot 2: Emissions scaling with parameters
ax2 = axes[1]
ax2.scatter(
    df_emissions["params_millions"],
    df_emissions["emissions_per_gen_mg"],
    s=100,
    color="forestgreen",
)
ax2.set_xlabel("Parameters (Millions)")
ax2.set_ylabel("CO2 Emissions per Generation (mg)")
ax2.set_title("Emissions Scaling with Model Size")

# Add labels
for _, row in df_emissions.iterrows():
    ax2.annotate(
        row["model"].split(" ")[0],
        (row["params_millions"], row["emissions_per_gen_mg"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=8,
    )

plt.tight_layout()
plt.show()

### Discussion: Environmental Impact

**Key observations:**
- Emissions scale roughly linearly (or worse) with model size
- A single generation has tiny emissions, but at scale it matters
- A model 4x larger doesn't necessarily give 4x better results

**Putting it in perspective:**
- 1g CO2 ≈ driving a car 4 meters
- Training GPT-3 emitted ~552 tonnes CO2 (≈ 120 cars/year)
- Inference costs dominate at scale (millions of queries per day)

**What can we do?**
- Choose appropriately-sized models for the task
- Use distilled models when possible
- Consider carbon-aware computing (run when grid is cleaner)
- Cache and reuse results

---

## Summary and Key Takeaways

In this practical, you learned about:

1. **Security Threats** (Exercise 1)
   - Tested prompt injection and jailbreak attacks
   - Implemented basic defenses (pattern detection, delimiters)
   - Understood limitations of simple defenses

2. **Bias Measurement & Steering** (Exercise 2)
   - Measured gender bias with CrowS-Pairs benchmark
   - Identified bias direction in activation space
   - Applied activation steering to reduce bias
   - Generated preference data for alignment

3. **Preference Learning** (Exercise 3)
   - Trained Bradley-Terry reward model on preference pairs
   - Explored DPO (Direct Preference Optimization) concepts
   - Compared steering vs training approaches

4. **Environmental Impact** (Exercise 4)
   - Measured CO2 emissions across model sizes
   - Understood trade-offs between performance and sustainability

### Key Tools & Concepts

| Topic | Tool/Method | Key Insight |
|-------|-------------|-------------|
| Security | Pattern matching, delimiters | No perfect defense exists yet |
| Bias measurement | CrowS-Pairs | Quantify stereotypical biases |
| Bias mitigation | Activation steering | Fast, no training needed |
| Alignment | Reward models, DPO | Learn from human preferences |
| Sustainability | codecarbon | Make environmental costs visible |

### Going Further

- **Red teaming**: Systematic adversarial testing at scale
- **Constitutional AI**: Training with ethical principles
- **Advanced interpretability**: Causal tracing, circuit analysis
- **Robust alignment**: Multi-stakeholder frameworks
- **Efficient inference**: Quantization, pruning, distillation